In [ ]:
import os, sys, json, glob, pickle, time
import numpy as np

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

ROOT = os.getcwd()
while not os.path.exists(os.path.join(ROOT, "pyproject.toml")):
    ROOT = os.path.dirname(ROOT)
sys.path.insert(0, ROOT)

RUNS = os.path.join(ROOT, "final-results", "wdbc")
FIGS = os.path.join(ROOT, "final-results", "figures")
CKPT = os.path.join(ROOT, "final-checkpoints", "wdbc")
for d in [RUNS, FIGS, CKPT]:
    os.makedirs(d, exist_ok=True)

RETRAIN = False
TAG = "wdbc_base"
SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]
GROUP, TAU, RESOLUTION, STEPS, BATCH, LR = 9, 3.0, 4, 20000, 64, 0.003
WIDTHS = [640, 640, 2 * GROUP]

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
Xraw, yall, names = data.data, data.target.astype(int), data.feature_names
print("WDBC", Xraw.shape, list(data.target_names), "malignant frac", round((yall == 0).mean(), 3))

In [ ]:
def split_indices(y, seed):
    rng = np.random.default_rng(seed)
    tr, va, te = [], [], []
    for c in [0, 1]:
        idx = rng.permutation(np.where(y == c)[0])
        n_tr, n_va = int(0.6 * len(idx)), int(0.2 * len(idx))
        tr += list(idx[:n_tr]); va += list(idx[n_tr:n_tr + n_va]); te += list(idx[n_tr + n_va:])
    return np.array(tr), np.array(va), np.array(te)

def quantile_transform(Xtr, X):

    S = np.sort(Xtr, axis=0)
    return (X[:, None, :] >= S[None, :, :]).sum(1) / len(S)

In [ ]:
def ternary_forward(circuit, X):
    h = X.astype(np.int8)
    for tt, conn in zip(circuit.truth_tables, circuit.connections):
        a = h[:, conn[:, 0]].astype(np.int32)
        b = h[:, conn[:, 1]].astype(np.int32)
        gather = np.arange(tt.shape[0], dtype=np.int64)[None, :] * 9 + (a + 1) * 3 + (b + 1)
        h = tt.reshape(-1)[gather].astype(np.int8)
    return h

In [ ]:
def run_seed(seed):
    import jax, jax.numpy as jnp, optax
    from pst_dtlgn.data.pipeline import TernaryPipeline
    from pst_dtlgn.network.topology import random_sparse
    from pst_dtlgn.network.network import PolynomialNetwork
    from pst_dtlgn.network.harden import harden_network_fast, TernaryLearnedCircuit
    from pst_dtlgn.binary_baseline import GroupSum
    from pst_dtlgn.training.trainer import train
    from pst_dtlgn.analysis.evaluation import evaluate_soft

    def loss_fn(preds, targets):
        logits = jax.vmap(gs)(preds)
        y = jax.nn.one_hot(targets.astype(jnp.int32).ravel(), 2)
        return jnp.mean(optax.sigmoid_binary_cross_entropy(logits, y))

    tr, va, te = split_indices(yall, seed)
    Xtr_r = quantile_transform(Xraw[tr], Xraw[tr])
    Xva_r = quantile_transform(Xraw[tr], Xraw[va])
    Xte_r = quantile_transform(Xraw[tr], Xraw[te])

    pipe = TernaryPipeline(resolution=RESOLUTION)
    Xtr = pipe.fit_transform(Xtr_r, mode="hard")
    Xva, Xte = pipe.transform(Xva_r, mode="hard"), pipe.transform(Xte_r, mode="hard")

    gs = GroupSum(k=2, tau=TAU)
    topo = random_sparse(jax.random.PRNGKey(seed), Xtr.shape[1], WIDTHS)
    net = PolynomialNetwork(jax.random.PRNGKey(seed + 2), topo)
    model, _ = train(net, optax.adam(LR), jnp.array(Xtr), jnp.array(yall[tr]),
                     total_steps=STEPS, batch_size=BATCH, lambda_max=0.1, lambda_gamma=2.0,
                     loss_fn=loss_fn, log_every=max(STEPS // 4, 100))

    soft = float(np.mean(evaluate_soft(model, jnp.array(Xte), gs) == yall[te]))
    hr = harden_network_fast(model)
    circuit = TernaryLearnedCircuit(hr)

    out = os.path.join(RUNS, TAG + "_seed" + str(seed))
    os.makedirs(out, exist_ok=True)
    res = {"seed": seed, "widths": WIDTHS, "group": GROUP, "tau": TAU, "pos_weight": 1.0,
           "resolution": RESOLUTION, "steps": STEPS, "soft_test": soft, "splits": {}}
    for split, X, idx in [("train", Xtr, tr), ("val", Xva, va), ("test", Xte, te)]:
        o = ternary_forward(circuit, np.asarray(X, dtype=np.int8)).reshape(len(X), 2, GROUP)
        np.save(os.path.join(out, split + "_trits.npy"), o)
        np.save(os.path.join(out, split + "_y.npy"), yall[idx])
        np.save(os.path.join(out, split + "_idx.npy"), idx)
        res["splits"][split] = {"forced_acc": float((o.sum(-1).argmax(1) == yall[idx]).mean()),
                                "n": len(idx)}
    ck = os.path.join(CKPT, TAG + "_seed" + str(seed))
    os.makedirs(ck, exist_ok=True)
    with open(os.path.join(ck, "harden.pkl"), "wb") as f:
        pickle.dump(hr, f)
    with open(os.path.join(out, "results.json"), "w") as f:
        json.dump(res, f, indent=2)
    return res

In [ ]:
if RETRAIN:
    for seed in SEEDS:
        t0 = time.time()
        r = run_seed(seed)
        print("seed", seed, "soft_test", round(r["soft_test"] * 100, 2),
              "forced_test", round(r["splits"]["test"]["forced_acc"] * 100, 2),
              "in", round(time.time() - t0), "s")

dirs = sorted(glob.glob(os.path.join(RUNS, TAG + "_seed*")))
print(len(dirs), "seed runs available")

In [ ]:
def commit_pred(o, T):
    s = o.sum(-1)
    v = np.where(s >= T, 1, np.where(s <= -T, -1, 0))
    c0 = (v[:, 0] == 1) & (v[:, 1] == -1)
    c1 = (v[:, 1] == 1) & (v[:, 0] == -1)
    return (c0 | c1), np.where(c1, 1, 0)

def load_split(d, split):
    return np.load(d + "/" + split + "_trits.npy"), np.load(d + "/" + split + "_y.npy")

In [ ]:
g = json.load(open(dirs[0] + "/results.json"))["group"]
rows = []
for T in range(1, g + 1):
    cov, acc, miss, deleg = [], [], [], []
    for d in dirs:
        o, y = load_split(d, "test")
        commit, pred = commit_pred(o, T)
        if commit.sum() == 0:
            continue
        mal = y == 0
        cov.append(commit.mean())
        acc.append((pred[commit] == y[commit]).mean())
        miss.append((commit & (pred == 1) & mal).sum() / mal.sum())
        deleg.append((~commit & mal).sum() / mal.sum())
    if cov:
        rows.append([T, np.mean(cov), np.mean(acc), np.std(acc, ddof=1) if len(acc) > 1 else 0.0,
                     np.mean(miss), np.mean(deleg)])
curve = np.array(rows)

forced_acc = np.mean([json.load(open(d + "/results.json"))["splits"]["test"]["forced_acc"] for d in dirs])
fm = []
for d in dirs:
    o, y = load_split(d, "test")
    fm.append(((o.sum(-1).argmax(1) == 1) & (y == 0)).sum() / (y == 0).sum())
forced_miss = np.mean(fm)

print("forced choice: acc", round(forced_acc * 100, 1), "malignant missed", round(forced_miss * 100, 1))
for T, c, a, _, mm, md_ in curve:
    print("T =", int(T), "coverage", round(c * 100, 1), "committed acc", round(a * 100, 1),
          "malignant missed", round(mm * 100, 1), "delegated", round(md_ * 100, 1))

In [ ]:
import matplotlib.pyplot as plt

def save(fig, name):
    for ext in ["svg", "pdf"]:
        fig.savefig(os.path.join(FIGS, name + "." + ext), bbox_inches="tight")

cov, acc, accsd = curve[:, 1] * 100, curve[:, 2] * 100, curve[:, 3] * 100

In [ ]:
fig, ax = plt.subplots(figsize=(4.6, 3.3))
ax.plot(cov, acc, "-o", color="tab:blue", ms=3, lw=1.4, label="ternary circuit (verdict readout)")
ax.fill_between(cov, acc - accsd, acc + accsd, color="tab:blue", alpha=0.15)
ax.axhline(forced_acc * 100, color="tab:red", ls="--", lw=1.2,
           label="forced choice (" + str(round(forced_acc * 100, 1)) + "%)")
ax.set_xlabel("coverage (% patients the circuit decides)")
ax.set_ylabel("accuracy on decided patients (%)")
ax.set_ylim(min(forced_acc * 100 - 1, acc.min() - 1), 100.5)
ax.legend(frameon=False, fontsize=8, loc="lower left")
ax.grid(alpha=0.25, lw=0.5)
fig.tight_layout()
save(fig, "wdbc_risk_coverage")

In [ ]:
fig, ax = plt.subplots(figsize=(4.6, 3.3))
ax.plot(cov, curve[:, 4] * 100, "-o", color="tab:purple", ms=3, lw=1.4,
        label="malignant missed (committed as benign)")
ax.axhline(forced_miss * 100, color="tab:red", ls="--", lw=1.2,
           label="forced choice misses " + str(round(forced_miss * 100, 1)) + "%")
ax.set_xlabel("coverage (% patients the circuit decides)")
ax.set_ylabel("malignant cases missed (%)")
ax.legend(frameon=False, fontsize=8, loc="upper left")
ax.grid(alpha=0.25, lw=0.5)
fig.tight_layout()
save(fig, "wdbc_malignant_miss")

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA

T_OP, K = 4, 15
knn = {"decided": [], "delegated": []}
lrm = {"decided": [], "delegated": []}

for d in dirs:
    o, _ = load_split(d, "test")
    idx = np.load(d + "/test_idx.npy")
    tr = np.load(d + "/train_idx.npy")
    commit, _ = commit_pred(o, T_OP)
    sc = StandardScaler().fit(Xraw[tr])
    Xtr_s, Xte_s = sc.transform(Xraw[tr]), sc.transform(Xraw[idx])

    d2 = ((Xte_s[:, None, :] - Xtr_s[None, :, :]) ** 2).sum(-1)
    lab = yall[tr][np.argsort(d2, axis=1)[:, :K]]
    frac = lab.mean(1)
    disagree = 1.0 - np.maximum(frac, 1.0 - frac)

    p = LogisticRegression(max_iter=2000).fit(Xtr_s, yall[tr]).predict_proba(Xte_s)[:, 1]
    margin = np.abs(p - 0.5)

    knn["decided"] += list(disagree[commit]); knn["delegated"] += list(disagree[~commit])
    lrm["decided"] += list(margin[commit]); lrm["delegated"] += list(margin[~commit])

for name, dd in [("kNN label disagreement", knn), ("logistic |p-0.5|", lrm)]:
    print(name, {k: (round(float(np.mean(v)), 3), round(float(np.std(v)), 3), len(v)) for k, v in dd.items()})

In [ ]:
faccs = [(d, json.load(open(d + "/results.json"))["splits"]["test"]["forced_acc"]) for d in dirs]
rep = min(faccs, key=lambda x: abs(x[1] - np.mean([f for _, f in faccs])))[0]
seed = json.load(open(rep + "/results.json"))["seed"]

o, y = load_split(rep, "test")
idx = np.load(rep + "/test_idx.npy")
commit, pred = commit_pred(o, T_OP)

sc_all = StandardScaler().fit(Xraw)
pca = PCA(2).fit(sc_all.transform(Xraw))
Z = pca.transform(sc_all.transform(Xraw[idx]))

fig, ax = plt.subplots(figsize=(5.0, 3.8))
for lab, nm, col in [(0, "malignant", "tab:red"), (1, "benign", "tab:green")]:
    m = (y == lab) & commit
    ax.scatter(Z[m, 0], Z[m, 1], s=18, c=col, alpha=0.7, label=nm + " (decided)")
ax.scatter(Z[~commit, 0], Z[~commit, 1], s=55, facecolors="none", edgecolors="black",
           linewidths=1.3, label="UNK (delegated)")
ax.set_xlabel("PC1 (" + str(round(pca.explained_variance_ratio_[0] * 100)) + "% var)")
ax.set_ylabel("PC2 (" + str(round(pca.explained_variance_ratio_[1] * 100)) + "% var)")
ax.set_title("WDBC test set (seed " + str(seed) + "), verdict at T=" + str(T_OP), fontsize=9)
ax.legend(frameon=False, fontsize=8, loc="upper right")
fig.tight_layout()
save(fig, "wdbc_pca")

In [ ]:
Xs = sc_all.transform(Xraw[idx])
unk = np.where(~commit)[0]
um, ub = unk[y[unk] == 0], unk[y[unk] == 1]
d2 = ((Xs[um][:, None, :] - Xs[ub][None, :, :]) ** 2).sum(-1)
i, j = np.unravel_index(d2.argmin(), d2.shape)
a, b = um[i], ub[j]

lr = LogisticRegression(max_iter=2000).fit(sc_all.transform(Xraw), yall)
pa, pb = lr.predict_proba(Xs[[a, b]])[:, 1]
print("coverage", round(commit.mean() * 100, 1), "unk malignant", len(um), "unk benign", len(ub))
print("nearest opposite-label delegated pair, squared distance", round(float(d2[i, j]), 2))
print("  patient A true malignant, logistic p(benign)", round(float(pa), 3))
print("  patient B true benign,    logistic p(benign)", round(float(pb), 3))
for k in np.argsort(np.abs(Xs[a] - Xs[b]))[:8]:
    print(" ", names[k], round(Xraw[idx[a], k], 3), round(Xraw[idx[b], k], 3),
          "z-diff", round(float(abs(Xs[a, k] - Xs[b, k])), 3))